In [1]:
import polars as pl
import datetime as dt
import os
from pathlib import Path

In [2]:
L2DATA_PATH = "/data/xujiayi/xjy/l2/test"

In [3]:
def normalize_date(date: dt.date | dt.datetime | str) -> str:
    if isinstance(date, (dt.datetime, dt.date)):
        return date.strftime("%Y%m%d")
    return str(date).replace("-", "").replace(".", "").strip("/")

In [4]:
date = normalize_date('20250930')

filepath = Path(L2DATA_PATH)/"raw"/date

outpath = Path(L2DATA_PATH)/"proc"/date
outpath.mkdir(parents=True, exist_ok=True)

In [5]:
sh = pl.read_parquet(filepath/'sh.pq').drop(['TradeMoney','LocalTime','SeqNo']).rename({
    'BizIndex':'ApplSeqNum',
    'Channel':'ChannelNo',
    'TickTime':'TransactTime',
    'Qty':'OrderQty'
}).with_columns([
    pl.col('TransactTime').str.to_time(format="%H:%M:%S%.3f"),
])
sh

ApplSeqNum,ChannelNo,SecurityID,TransactTime,Type,BuyOrderNO,SellOrderNO,Price,OrderQty,TickBSFlag
i64,i64,i64,time,str,i64,i64,f64,i64,str
1,1,751028,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
2,1,751900,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
3,1,751004,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
4,1,751019,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
5,1,751992,06:00:00.110,"""S""",0,0,0.0,0,"""SUSP"""
…,…,…,…,…,…,…,…,…,…
28262536,6,560820,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""
28262537,6,588130,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""
28262538,6,603210,15:05:00.560,"""S""",0,0,0.0,0,"""ENDTR"""


In [78]:
shcd = sh.filter(pl.col('Type')=='D').drop(['ApplSeqNum','Type','TickBSFlag'])
shcd

ChannelNo,SecurityID,TransactTime,BuyOrderNO,SellOrderNO,Price,OrderQty
i64,i64,time,i64,i64,f64,i64
2,688545,09:15:11.450,0,56334,38.17,12717
2,688545,09:15:17.280,0,192292,37.49,267
2,688545,09:15:18.380,0,202969,45.23,350
2,688545,09:15:28.010,225350,0,37.7,1325
2,688545,09:15:29.850,0,228504,38.0,778
…,…,…,…,…,…,…
6,516510,14:59:59.930,17255333,0,1.673,280300
6,516510,14:59:59.930,17167135,0,1.67,280300
6,513700,14:59:59.950,16136535,0,0.77,3000


In [60]:
shwt_added = sh.filter(pl.col('Type')=='A').drop('Type').with_columns([
    pl.when(pl.col('BuyOrderNO')!=0).then(pl.col('BuyOrderNO')).otherwise(pl.col('SellOrderNO')).alias('OrderNO'),
    pl.when(pl.col('TickBSFlag')=='B').then(pl.lit(1)).otherwise(pl.lit(-1)).alias('Side')
]).drop(['BuyOrderNO','SellOrderNO','TickBSFlag', 'ApplSeqNum'])
shwt_added 

ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side
i64,i64,time,f64,i64,i64,i32
2,688545,09:15:01.970,36.01,300,12330,1
2,688545,09:15:02.550,41.42,1704,20176,-1
2,688545,09:15:02.550,45.23,9,21026,-1
2,688545,09:15:02.550,37.02,1000,21338,1
2,688545,09:15:02.680,36.71,1000,21373,1
…,…,…,…,…,…,…
5,603202,14:59:42.510,98.73,100,19318579,-1
5,603202,14:59:45,96.87,100,19320909,1
5,603202,14:59:52.550,96.77,200,19329054,-1


In [70]:
shcj = sh.filter(pl.col('Type')=='T').drop(['ApplSeqNum','Type','TickBSFlag'])

cj_df = shcj.filter(~((pl.col("TransactTime") <= pl.time(9, 25)) | (pl.col("TransactTime") >= pl.time(14, 57))))
cj_df

ChannelNo,SecurityID,TransactTime,BuyOrderNO,SellOrderNO,Price,OrderQty
i64,i64,time,i64,i64,f64,i64
2,688545,09:25:00.010,322746,203130,38.75,50
2,688545,09:25:00.010,322746,282462,38.75,393
2,688545,09:25:00.010,322746,300023,38.75,200
2,688545,09:25:00.010,322746,301776,38.75,157
2,688545,09:25:00.010,336104,301776,38.75,1000
…,…,…,…,…,…,…
2,600581,14:56:59.990,18661006,18565587,4.34,300
2,600581,14:56:59.990,18661006,18565604,4.34,100
2,600581,14:56:59.990,18661006,18565607,4.34,100


In [71]:
cj_buy = cj_df.select([
    "ChannelNo", "BuyOrderNO", "SecurityID", "OrderQty", "Price", "TransactTime",
]).rename({"BuyOrderNO": "OrderNO"}).with_columns(pl.lit(1).alias("Side"))
cj_sell = cj_df.select([
    "ChannelNo", "SellOrderNO", "SecurityID", "OrderQty", "Price", "TransactTime",
]).rename({"SellOrderNO": "OrderNO"}).with_columns(pl.lit(-1).alias("Side"))
cj_all = pl.concat([cj_buy, cj_sell])
cj_summary = (
    cj_all
    .sort(["ChannelNo", "OrderNO", "SecurityID", "Side", "TransactTime"])
    .group_by(["ChannelNo", "OrderNO", "SecurityID", "Side"])
    .agg([
        pl.sum("OrderQty"),
        pl.last("Price"),  # 一笔主动单可能同时吃掉多笔挂单
        pl.last("TransactTime"),

    ])
)
cj_summary

ChannelNo,OrderNO,SecurityID,Side,OrderQty,Price,TransactTime
i64,i64,i64,i32,i64,f64,time
6,16522274,601012,-1,1100,17.97,14:49:53.960
2,2293407,560780,1,1300,1.735,09:37:25.710
5,13881025,601696,-1,2400,14.45,13:29:31.080
1,8820661,600036,1,100,40.46,14:20:41.010
4,7211056,600489,1,3600,22.1,13:02:19.710
…,…,…,…,…,…,…
4,4596789,600100,1,6000,8.07,10:26:36.210
3,6494523,600171,-1,100,36.87,10:09:30.760
5,424145,600403,-1,3000,4.2,13:44:48.940


In [72]:
# 部分成交：同时存在于委托和成交（inner join）
partial = shwt_added.join(
    cj_summary.select(["ChannelNo", "OrderNO", "SecurityID", "Side", "OrderQty"]),
    on=["ChannelNo", "OrderNO", "SecurityID", "Side"],
    how="inner"
).with_columns([
    (pl.col("OrderQty") + pl.col("OrderQty_right")).alias("OrderQty"),
    pl.lit("主动部分成交").alias("OrderStatus")
]).drop("OrderQty_right")
partial

ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side,OrderStatus
i64,i64,time,f64,i64,i64,i32,str
2,688545,09:15:02.680,39.9,3778,23083,-1,"""主动部分成交"""
2,688545,09:15:02.920,37.0,1606,25945,-1,"""主动部分成交"""
2,688545,09:15:02.920,39.85,6000,26050,-1,"""主动部分成交"""
2,688545,09:15:03.210,39.49,4000,29274,-1,"""主动部分成交"""
2,688545,09:15:03.350,40.85,400,31336,-1,"""主动部分成交"""
…,…,…,…,…,…,…,…
6,601766,14:56:59.920,7.47,4900,17133703,-1,"""主动部分成交"""
6,603200,14:56:59.940,90.78,400,17133717,1,"""主动部分成交"""
6,688615,14:56:59.990,159.82,3000,17133768,1,"""主动部分成交"""


In [73]:
# 未成交：存在于委托但不存在于成交（anti join）
untouched = shwt_added.join(
    cj_summary.select(["ChannelNo", "OrderNO", "SecurityID", "Side"]),
    on=["ChannelNo", "OrderNO", "SecurityID", "Side"],
    how="anti"
).with_columns(
    pl.lit("挂单被动成交").alias("OrderStatus")
)
untouched = untouched.select(partial.columns)
untouched


ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side,OrderStatus
i64,i64,time,f64,i64,i64,i32,str
2,688545,09:15:01.970,36.01,300,12330,1,"""挂单被动成交"""
2,688545,09:15:02.550,41.42,1704,20176,-1,"""挂单被动成交"""
2,688545,09:15:02.550,45.23,9,21026,-1,"""挂单被动成交"""
2,688545,09:15:02.550,37.02,1000,21338,1,"""挂单被动成交"""
2,688545,09:15:02.680,36.71,1000,21373,1,"""挂单被动成交"""
…,…,…,…,…,…,…,…
5,603202,14:59:42.510,98.73,100,19318579,-1,"""挂单被动成交"""
5,603202,14:59:45,96.87,100,19320909,1,"""挂单被动成交"""
5,603202,14:59:52.550,96.77,200,19329054,-1,"""挂单被动成交"""


In [74]:
new = cj_summary.join(
    shwt_added.select(["ChannelNo", "OrderNO", "SecurityID", "Side"]),
    on=["ChannelNo", "OrderNO", "SecurityID", "Side"],
    how="anti"
).with_columns([
    pl.lit("主动完全成交").alias("OrderStatus"),
])
new = new.select(partial.columns)
new

ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side,OrderStatus
i64,i64,time,f64,i64,i64,i32,str
6,601012,14:49:53.960,17.97,1100,16522274,-1,"""主动完全成交"""
2,560780,09:37:25.710,1.735,1300,2293407,1,"""主动完全成交"""
5,601696,13:29:31.080,14.45,2400,13881025,-1,"""主动完全成交"""
4,600489,13:02:19.710,22.1,3600,7211056,1,"""主动完全成交"""
3,600478,14:27:40.210,6.59,2100,16546876,1,"""主动完全成交"""
…,…,…,…,…,…,…,…
1,600206,09:30:18.240,21.76,5000,584946,1,"""主动完全成交"""
5,515190,14:09:10.630,1.547,1000,16038377,1,"""主动完全成交"""
6,600023,14:52:21.390,4.95,100,16740458,1,"""主动完全成交"""


In [89]:
shwt = pl.concat([partial, untouched, new])
shwt.filter(pl.col('OrderNO')==4004189).filter(pl.col('SecurityID')==688545)

ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side,OrderStatus
i64,i64,time,f64,i64,i64,i32,str
2,688545,09:48:38.470,39.9,5100,4004189,1,"""主动完全成交"""


In [80]:
shwt = pl.concat([partial, untouched, new]).drop('OrderStatus')
shwt.sort(['ChannelNo','OrderNO','SecurityID','Side'])
shwt

ChannelNo,SecurityID,TransactTime,Price,OrderQty,OrderNO,Side
i64,i64,time,f64,i64,i64,i32
1,600383,09:15:00.010,4.51,12900,1,-1
1,600518,09:15:00.010,2.01,666200,2,1
1,600567,09:15:00.010,1.82,910000,3,-1
1,600567,09:15:00.010,1.83,910000,4,-1
1,600567,09:15:00.010,1.77,910000,5,1
…,…,…,…,…,…,…
20,900908,14:59:57.240,0.67,20000,35151,-1
20,900938,14:59:57.590,0.277,500,35152,1
20,900926,14:59:58.380,1.126,33200,35153,1


In [81]:
orders = (
    shwt
    .select([
        "ChannelNo",
        "SecurityID",
        "OrderNO",
        "Price",
        "OrderQty",
        "Side",
    ])
)

cj = pl.concat([shcj, shcd])

# 2. 买方订单成交扣减
buy_filled = (
    cj
    .select([
        "ChannelNo",
        "SecurityID",
        pl.col("BuyOrderNO").alias("OrderNO"),
        pl.col("OrderQty").alias("DealQty"),
    ])
    .with_columns(pl.lit(1).alias('Side'))
)

# 3. 卖方订单成交扣减
sell_filled = (
    cj
    .select([
        "ChannelNo",
        "SecurityID",
        pl.col("SellOrderNO").alias("OrderNO"),
        pl.col("OrderQty").alias("DealQty"),
    ])
    .with_columns(pl.lit(-1).alias('Side'))
)

# 4. 每张订单的累计成交数量
filled = (
    pl.concat([buy_filled, sell_filled])
    .group_by(["ChannelNo", "SecurityID", "Side", "OrderNO"])
    .agg(
        pl.col("DealQty").sum().alias("FilledQty")
    )
)

# 8. 计算每张订单剩余数量
alive_orders = (
    orders
    .join(
        filled,
        on=["ChannelNo", "SecurityID", "Side", "OrderNO"],
        how="left",
    )
    .with_columns([
        (
            pl.col("OrderQty")
            - pl.col("FilledQty").fill_null(0)
        ).alias("RemainQty")
    ])
    .filter(pl.col("RemainQty") > 0)
)

# 9. 聚合成价格档位
price_level = (
    alive_orders
    .group_by(["SecurityID", "Side", "Price"])
    .agg(
        pl.col("RemainQty").sum().alias("Qty")
    )
)

# 10. 买盘：价格从高到低
bid_book = (
    price_level
    .filter(pl.col("Side") == 1)
    .with_columns(
        pl.col("Price")
        .rank(method="dense", descending=True)
        .over("SecurityID")
        .cast(pl.Int32)
        .alias("Level")
    )
    .filter(pl.col("Level") <= 10)
    .sort(["SecurityID", "Level"])
    .rename({
        "Price": "BidPrice",
        "Qty": "BidQty",
    })
    .select(["SecurityID", "Level", "BidPrice", "BidQty"])
)

# 11. 卖盘：价格从低到高
ask_book = (
    price_level
    .filter(pl.col("Side") == -1)
    .with_columns(
        pl.col("Price")
        .rank(method="dense", descending=False)
        .over("SecurityID")
        .cast(pl.Int32)
        .alias("Level")
    )
    .filter(pl.col("Level") <= 10)
    .sort(["SecurityID", "Level"])
    .rename({
        "Price": "AskPrice",
        "Qty": "AskQty",
    })
    .select(["SecurityID", "Level", "AskPrice", "AskQty"])
)

# 12. 合并买卖盘
close_book = (
    bid_book
    .join(
        ask_book,
        on=["SecurityID", "Level"],
        how="full",
        coalesce=True,
    )
    .sort(["SecurityID", "Level"])
)

close_book

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
501001,1,1.497,3700,1.482,6900
501001,2,1.493,28602,1.49,400
501001,3,1.487,3600,1.493,24600
501001,4,1.486,500,1.496,100
501001,5,1.482,800,1.501,400
…,…,…,…,…,…
900948,6,1.937,17200,1.913,35000
900948,7,1.936,34500,1.914,27200
900948,8,1.935,14300,1.915,61800


In [82]:
close_book.filter(pl.col('SecurityID')==600000)

SecurityID,Level,BidPrice,BidQty,AskPrice,AskQty
i64,i32,f64,i64,f64,i64
600000,1,13.27,27700,10.85,12600
600000,2,13.26,2000,11.22,6000
600000,3,13.22,2000,11.46,31400
600000,4,12.8,1000,11.49,2700
600000,5,12.66,13100,11.5,300
600000,6,12.6,41900,11.55,4700
600000,7,12.54,4600,11.7,300
600000,8,12.51,800,11.8,83000
600000,9,12.42,2800,11.82,146600
